In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error
from scipy.special import gamma

In [ ]:
# Finite-Horizon LQR Function
def finite_horizon_lqr(A, B, Q, R, Q_f, T):
    n, m = B.shape
    P_list = [None] * (T + 1)
    K_list = [None] * T
    P_list[T] = Q_f
    for t in range(T - 1, -1, -1):
        P_next = P_list[t + 1]
        K_t = np.linalg.inv(R + B.T @ P_next @ B) @ (B.T @ P_next @ A)
        P_t = Q + A.T @ P_next @ (A - B @ K_t)
        K_list[t] = K_t
        P_list[t] = P_t
    return K_list, P_list

# Simulation Function
def simulate_finite_horizon_lqr(A, B, K_list, x0, T):
    n, m = B.shape
    X = np.zeros((n, T + 1))
    U = np.zeros((m, T))
    X[:, [0]] = x0
    for t in range(T):
        U[:, [t]] = -K_list[t] @ X[:, [t]]
        X[:, [t + 1]] = A @ X[:, [t]] + B @ U[:, [t]]
    return X.T, U.T

In [ ]:
# Function to construct the psi matrix
def psi_matrix(alpha, j):
    """
    Constructs the diagonal matrix psi(alpha, j) based on the provided formula.
    """
    diagonal_elements = [
        gamma(j - alpha[i]) / (gamma(-alpha[i]) * gamma(j + 1)) for i in range(len(alpha))
    ]
    return np.diag(diagonal_elements)

# Function to generate data for a single trial
def generate_data_single_trial(p, A, B, alpha, x_0, u_0, j=1):
    """
    Generates x_0, u_0, and x_1 for a single trial.
    Args:
        p (int): Number of initial conditions.
        A (np.ndarray): System matrix.
        B (np.ndarray): Input matrix.
        C_alpha (np.ndarray): Diagonal matrix derived from alpha.
        alpha (np.ndarray): Diagonal elements of C_alpha.
        j (int): Time step for psi computation (default is 1).
        x_0 (np.ndarray): Initial conditions (p x n).
        u_0 (np.ndarray): Control inputs (p x n).
    
    Returns:
        x_1 (np.ndarray): Generated states at the second time step (p x n).
    """
    n = B.shape[0]

    #x_0 = np.random.rand(p, n)  # Initial conditions
    #u_0 = np.random.rand(p, m)  # Control inputs
    w_0 = np.random.randn(p, n)  # Gaussian noise
    #w_0 = np.zeros((p,n))
    x_1 = np.zeros((p, n))  # Placeholder for x_1

    # Compute x_1 for each initial condition
    psi_diag = psi_matrix(alpha, j)  # Compute psi for time step j
    for i in range(p):
        x_1[i, :] = (
            np.dot(A, x_0[i, :]) 
            + np.dot(B, u_0[i, :]) 
            - np.dot(psi_diag, x_0[i, :]) 
            + w_0[i, :]
        )
    
    return x_0, u_0, x_1

# Function to generate data for multiple trials
def generate_data_multiple_trials(N, p, A, B, alpha, x_0, u_0):
    """
    Generates x_0, u_0, and x_1 across multiple trials.
    Args:
        N (int): Number of trials.
        p (int): Number of initial conditions.
        A (np.ndarray): System matrix.
        B (np.ndarray): Input matrix.
        alpha (np.ndarray): Diagonal elements of C_alpha.
    
    Returns:
        x_0_all (np.ndarray): All initial conditions (N x p x n).
        u_0_all (np.ndarray): All control inputs (N x p x n).
        x_1_all (np.ndarray): All generated states (N x p x n).
    """
    

    # Storage for all trials
    x_0_all = []
    u_0_all = []
    x_1_all = []

    for trial in range(N):
        x_0, u_0, x_1 = generate_data_single_trial(p, A, B, alpha, x_0, u_0)
        x_0_all.append(x_0)
        u_0_all.append(u_0)
        x_1_all.append(x_1)
    
    # Convert to numpy arrays
    x_0_all = np.array(x_0_all)
    u_0_all = np.array(u_0_all)
    x_1_all = np.array(x_1_all)
    
    return x_0_all, u_0_all, x_1_all

# Function to construct the block diagonal matrix (π or φ)
def construct_block_diagonal_matrix(data, n):
    """
    Constructs a block diagonal matrix for π or φ.
    Args:
        data (np.ndarray): Input data (x_0 or u_0) of shape (p, n).
        n (int): state
    
    Returns:
        np.ndarray: Block diagonal matrix of shape (p * n, n).
    """
    m = data.shape[2]
    p = data.shape[1]  # Number of initial conditions
    matrix = np.zeros((n * p, n * m))

    for i in range(p):
        for j in range(n):
            matrix[i * n + j, j * m : (j + 1) * m] = data[0, i]  
    return matrix

# Function to construct the diagonal matrix Ω
def construct_diagonal_matrix(data):
    """
    Constructs a diagonal matrix Ω from x_0.
    Args:
        data (np.ndarray): Input data (x_0) of shape (p, n).
    
    Returns:
        np.ndarray: Diagonal matrix of shape (p * n, n).
    """
    p, n = data[0].shape
    diagonal_matrix = np.zeros((p * n, n))
    for i in range(p):
        diagonal_matrix[i * n:(i + 1) * n, :] = np.diag(data[0, i])
    return diagonal_matrix

# Function to solve the least squares problem
def solve_least_squares(X, x_0, u_0):
    """
    Solves the least squares problem described in the equations.
    Args:
        X (np.ndarray): Target data matrix of shape (N, p, n).
        x_0 (np.ndarray): Initial conditions (N, p, n).
        u_0 (np.ndarray): Control inputs (N, p, m).

    Returns:
        theta_hat (np.ndarray): Estimated parameter vector [gamma, beta, alpha].
    """
    n = x_0.shape[2]
    m = u_0.shape[2]

    # Construct π, φ, and Ω
    pi = construct_block_diagonal_matrix(x_0, n)
    phi = construct_block_diagonal_matrix(u_0, n)
    #omega = construct_diagonal_matrix(x_0)

    # Construct the matrix ξ
    xi = np.hstack([pi, phi])  # Shape: (p * n, 3 * n)
    #print(xi)
    #print(xi.T @ xi)
    #print((xi.T @ xi).shape)
    # Solve the least squares problem
    theta_hat = np.linalg.inv(xi.T @ xi) @ xi.T @ np.mean(X, axis = 0).reshape(-1)

    
    # Reshape theta_hat into A_hat and B_hat
    A_alpha_hat = theta_hat[:n**2].reshape(n, n)
    B_hat = theta_hat[n**2:].reshape(n, m)

    return A_alpha_hat, B_hat
    #return theta_hat


# Function to solve the least squares problem known A and \alpha
def solve_least_squares_know_A_alpha(X, x_0, u_0, A, alpha):
    """
    Solves the least squares problem described in the equations.
    Args:
        X (np.ndarray): Target data matrix of shape (N, p, n).
        x_0 (np.ndarray): Initial conditions (N, p, n).
        u_0 (np.ndarray): Control inputs (N, p, m).

    Returns:
        theta_hat (np.ndarray): Estimated parameter vector [gamma, beta, alpha].
    """
    n = x_0.shape[2]
    m = u_0.shape[2]

    # Construct π, φ, and Ω
    pi = construct_block_diagonal_matrix(x_0, n)
    phi = construct_block_diagonal_matrix(u_0, n)
    omega = construct_diagonal_matrix(x_0)

    pi_A = pi @ A.reshape(-1)
    Omega_alpha = omega @ alpha


    # Construct the matrix ξ
    xi = np.hstack([phi])  # Shape: (p * n, 3 * n)
    #print(xi)
    #print(xi.T @ xi)
    #print((xi.T @ xi).shape)
    # Solve the least squares problem
    theta_hat = np.linalg.inv(xi.T @ xi) @ xi.T @ (np.mean(X, axis = 0).reshape(-1) - pi_A - Omega_alpha)

    
    # Reshape theta_hat into A_hat and B_hat
    #A_alpha_hat = theta_hat[:n**2].reshape(n, n)
    #B_hat = theta_hat[n**2:].reshape(n, m)

    #return A_alpha_hat, B_hat
    return theta_hat

def replace_diagonal_and_compute_difference(A_alpha_hat, diagonal_A):
    """
    Replaces the diagonal elements of A_alpha_hat with diagonal_A 
    and computes the difference between the original and new diagonal elements.

    Parameters:
    A_alpha_hat (numpy.ndarray): An n x n matrix.
    diagonal_A (numpy.ndarray): An n x 1 vector.

    Returns:
    tuple: A_hat (n x n matrix), alpha_hat (n x 1 vector).
           A_hat is A_alpha_hat with its diagonal replaced by diagonal_A.
           alpha_hat is the difference between the original and new diagonal elements.
    """
    # Validate inputs
    n = A_alpha_hat.shape[0]
    if A_alpha_hat.shape[0] != A_alpha_hat.shape[1]:
        raise ValueError("A_alpha_hat must be an n x n square matrix.")
    if diagonal_A.shape[0] != n or diagonal_A.shape[1] != 1:
        raise ValueError("diagonal_A must be an n x 1 vector.")

    # Extract original diagonal elements of A_alpha_hat
    original_diagonal = np.diag(A_alpha_hat)

    # Compute alpha_hat as the difference between original and new diagonal
    alpha_hat = (original_diagonal - diagonal_A.flatten()).reshape(-1, 1)

    # Replace the diagonal of A_alpha_hat with diagonal_A
    A_hat = A_alpha_hat.copy()
    np.fill_diagonal(A_hat, diagonal_A.flatten())

    return A_hat, alpha_hat

def extract_diagonal(A):
    """
    Extracts the diagonal elements of a square matrix A and forms a column vector.

    Parameters:
    A (numpy.ndarray): An n x n matrix.

    Returns:
    numpy.ndarray: An n x 1 vector containing the diagonal elements of A.
    """
    # Validate input
    if A.shape[0] != A.shape[1]:
        raise ValueError("Input matrix A must be square (n x n).")
    
    # Extract the diagonal elements and reshape into a column vector
    diagonal_vector = np.diag(A).reshape(-1, 1)
    
    return diagonal_vector

# Define the ψ(α_i, j) function
def psi(alpha_i, j):
    """
    Compute ψ(α_i, j) = Γ(j - α_i) / (Γ(-α_i) Γ(j + 1)).

    Parameters:
    - alpha_i: Scalar value of α_i (float).
    - j: Scalar index j (integer).

    Returns:
    - ψ(α_i, j): Computed value (float).
    """
    return gamma(j - alpha_i) / (gamma(-alpha_i) * gamma(j + 1))

# Define the diagonal matrix D(α, j)
def D(alpha, j):
    """
    Compute the diagonal matrix D(α, j).

    Parameters:
    - alpha: List or numpy array of α values.
    - j: Index j (integer).

    Returns:
    - D: Diagonal matrix (numpy array).
    """
    n = len(alpha)
    diag_elements = [psi(ai, j) for ai in alpha]
    return np.diag(diag_elements)

# Define A_j based on the given formula
def A_j(alpha, j, A):
    """
    Compute A_j based on the given formula:
    A_j = A - diag(α_1, ..., α_n) for j = 0
    A_j = -D(α, j + 1) for j >= 1

    Parameters:
    - alpha: List or numpy array of α values.
    - j: Index j (integer).
    - A: Matrix A (numpy array).

    Returns:
    - A_j: Computed matrix A_j (numpy array).
    """
    if j == 0:
        diag_matrix = np.diag(alpha)  # Create diag(α_1, ..., α_n)
        return A - diag_matrix
    else:
        return -D(alpha, j + 1)
    

# (A_0^T)
def A_j_T(alpha, j, A):
    """
    Compute A_j based on the given formula:
    A_j = A - diag(α_1, ..., α_n) for j = 0
    A_j = -D(α, j + 1) for j >= 1

    Parameters:
    - alpha: List or numpy array of α values.
    - j: Index j (integer).
    - A: Matrix A (numpy array).

    Returns:
    - A_j: Computed matrix A_j (numpy array).
    """
    if j == 1:
        diag_matrix = np.diag(alpha)  # Create diag(α_1, ..., α_n)
        return (A - diag_matrix).T
    else:
        return -D(alpha, j)
    
def A_j_list(alpha, A, T):
    """
    Generate a list of A_j^T matrices for j = 0 to T-1.

    Parameters:
    - alpha: List or numpy array of α values.
    - T: Total number of A_j matrices to compute.
    - A: Matrix A (numpy array).

    Returns:
    - A_j_list: List of A_j^T matrices.
    """
    return [A_j_T(alpha, j+1, A) for j in range(T)]
'''''''''
def A_mlist(alpha, A, T):
    """
    Generate a list of A_j^T matrices for j = 0 to T-1.

    Parameters:
    - alpha: List or numpy array of α values.
    - T: Total number of A_j matrices to compute.
    - A: Matrix A (numpy array).

    Returns:
    - A_j_list: List of A_j^T matrices.
    """
    return [A_j(alpha, j, A) for j in range(T)]
'''''''''''
# Define G_k using recursion
def G_k(alpha, A, max_k):
    """
    Compute G_k recursively based on the given formula:
    G_k = I for k = 0
    G_k = Σ_{j=0}^{k-1} A_j G_{k-1-j} for k >= 1

    Parameters:
    - alpha: List or numpy array of α values.
    - k: Current recursion depth (integer).
    - A: Matrix A (numpy array).
    - max_k: Maximum k for recursion.

    Returns:
    - G_list: List of G_k matrices up to G_max_k (list of numpy arrays).
    """
    n = len(alpha)
    I = np.eye(n)  # Identity matrix
    G_list = [I]  # Initialize G_0 as I

    for current_k in range(1, max_k + 1):
        G_k = np.zeros_like(A)  # Initialize G_k to zero
        for j in range(current_k):
            A_j_matrix = A_j(alpha, j, A)
            G_k += A_j_matrix @ G_list[current_k - 1 - j]  # Recursive formula
        G_list.append(G_k)

    return G_list

def create_G_lambda(T, Q, B, R, A_j_list, G_list):
    """
    Generate the G_lambda matrix as shown in the provided image.

    Parameters:
    - T: int, time horizon
    - Q: numpy array, a square matrix
    - B: numpy array, input matrix
    - R: numpy array, a square matrix
    - A_j_list
    - G_list

    Returns:
    - G_lambda: numpy array, the generated G_lambda matrix
    """
    # Get dimensions
    n, m = B.shape
    BR_inv_B_T = B @ np.linalg.inv(R) @ B.T 

    # Initialize the matrix
    G_lambda = np.zeros((T * n, T * n))

    # Fill in the blocks
    for row in range(T):
        for col in range(T):
            if row > col :
                G_lambda[row * n:(row + 1) * n, col * n:(col + 1) * n] = -Q @ G_list[row - col] @ BR_inv_B_T
            elif row == col:
                G_lambda[row * n:(row + 1) * n, col * n:(col + 1) * n] = -Q @ BR_inv_B_T               #G_0 = I
            elif row < col:
                G_lambda[row * n:(row + 1) * n, col * n:(col + 1) * n] = A_j_list[col - row - 1]
    
    return G_lambda

def create_H_lambda(T, Q, G_list):
    """
    Generate the H_lambda matrix as shown in the provided structure.

    Parameters:
    - T: int, time horizon (number of blocks in the matrix).
    - Q: numpy array, a square matrix.
    - G_list

    Returns:
    - H_lambda: numpy array, the generated H_lambda matrix.
    """
    # Get the dimensions of Q
    n = Q.shape[0]

    # Initialize the H_lambda matrix with zeros
    H_lambda = np.zeros((T * n, n))

    # Fill in the rows of H_lambda
    for i in range(T):
        G_i = G_list[i + 1]  # Compute G_i (1-based index)
        H_lambda[i * n:(i + 1) * n, :] = Q @ G_i

    return H_lambda

def compute_lambda(G_lambda, H_lambda, x_0):
    # Identity matrix of the same size as G_lambda
    I = np.eye(G_lambda.shape[0])

    # Compute (I - G_lambda)^(-1)
    inv_I_minus_G_lambda = np.linalg.inv(I - G_lambda)

    # Compute λ
    lambda_vector = 2 * inv_I_minus_G_lambda @ H_lambda @ x_0

    return lambda_vector

def compute_u_with_block_matrix(R, B, G_lambda, H_lambda, x0, T):
    """
    Compute u using the formula:
    u = -[R^(-1) B^T, R^(-1) B^T, ..., R^(-1) B^T] @ ((I - G_lambda)^(-1) H_lambda x0)

    Parameters:
    - R: numpy array, square matrix R (assumed invertible).
    - B: numpy array, matrix B.
    - G_lambda: numpy array, square matrix G_lambda.
    - H_lambda: numpy array, matrix H_lambda.
    - x0: numpy array, vector x0.
    - T: int, number of repetitions of the block matrix.

    Returns:
    - u: numpy array, the computed vector u.
    """
    # Compute R^(-1)
    R_inv = np.linalg.inv(R)

    # Compute R^(-1) B^T
    R_inv_B_T = R_inv @ B.T

    # Construct the block matrix [R^(-1) B^T, R^(-1) B^T, ..., R^(-1) B^T] (T times)
    I_T = np.eye(T)
    block_diag_matrix = np.kron(I_T, R_inv_B_T)

    # Compute (I - G_lambda)^(-1)
    I = np.eye(G_lambda.shape[0])
    inv_I_minus_G = np.linalg.inv(I - G_lambda)

    # Compute (I - G_lambda)^(-1) * H_lambda * x0
    term = inv_I_minus_G @ H_lambda @ x0

    # Compute u
    u_vector = -block_diag_matrix @ term
    return u_vector
# Simulate the fractional-order system
def simulate_fractional_system_no_noise(A, B, alpha, x0, u, steps):
    n = A.shape[0]
    x = np.zeros((steps + 1, n))
    x[0] = x0
    
    # Calculate system response over time
    for k in range(steps):
        summation_term = sum(A_j(alpha, j, A) @ x[k - j] for j in range(k + 1))
        x[k + 1] = summation_term + B @ u[k]
        #x[k + 1] = summation_term + B @ u[k] 
    
    return x

In [ ]:
LQR_Q_array = np.load('/Users/leskywalker/Documents/ICML2025/FODSData/LQR_Q.npy')
LQR_R_array = np.load('/Users/leskywalker/Documents/ICML2025/FODSData/LQR_R.npy')
optimal_control_U = np.load('/Users/leskywalker/Documents/ICML2025/FODSData/optimal_control_U.npy')
trajectories_array = np.load('/Users/leskywalker/Documents/ICML2025/FODSData/fractional_system_trajectories.npy')
inputs_array = np.load('/Users/leskywalker/Documents/ICML2025/FODSData/fractional_system_inputs.npy')
# Load system matrices and fractional orders
A = np.load('/Users/leskywalker/Documents/ICML2025/FODSData/system_matrix_A.npy')
B = np.load('/Users/leskywalker/Documents/ICML2025/FODSData/system_matrix_B.npy')
alpha = np.load('/Users/leskywalker/Documents/ICML2025/FODSData/fractional_orders_alpha.npy')


In [ ]:
trajectories_array_train = trajectories_array[:3000]
inputs_array_train = inputs_array[:3000]
trajectories_array_test = trajectories_array[9999]
inputs_array_test = inputs_array[9999]
Q_test = LQR_Q_array[9999]
R_test = LQR_R_array[9999]
Time = inputs_array.shape[1]
optimal_control_U_test = optimal_control_U[9999]

In [ ]:
def identify_lti_system_ols(trajectories_array, inputs_array):
    """Identify LTI system matrices A and B using OLS across multiple trajectories.
    trajectories_array: Array of state trajectories, shape (N, T+1, dimension_x)
    inputs_array: Array of input trajectories, shape (N, T, dimension_u)
    """
    _, _, dimension_x = trajectories_array.shape
    _, _, dimension_u = inputs_array.shape

    # Reshape trajectories and inputs for regression
    X_t = trajectories_array[:, :-1, :].reshape(-1, dimension_x).T  # x_k
    X_t1 = trajectories_array[:, 1:, :].reshape(-1, dimension_x).T  # x_{k+1}
    U_t = inputs_array.reshape(-1, dimension_u).T  # u_k
    #print(U_t.shape)

    # Construct regression matrix Phi
    Phi = np.vstack((X_t, U_t))

    # Solve for Theta using least squares
    Theta = X_t1 @ np.linalg.pinv(Phi)

    # Extract A and B matrices
    A_est = Theta[:, :dimension_x]
    B_est = Theta[:, dimension_x:]

    return A_est, B_est

In [ ]:
A_est, B_est = identify_lti_system_ols(trajectories_array_train, inputs_array_train)
K_L, P_L = finite_horizon_lqr(A_est, B_est, Q_test, R_test, Q_test, Time)
trajectory_hat_LTI, U_hat_LTI = simulate_finite_horizon_lqr(A_est, B_est, K_L, trajectories_array_test[0].reshape(trajectories_array_test.shape[1],1), Time)

In [ ]:
import seaborn as sns

T, n = U_hat_LTI.shape

# Define plot parameters
fig, ax = plt.subplots( 1, n, figsize=(15, 3), dpi=300, sharex=True, sharey=True)

# Define colors for trajectories
colors_true = sns.color_palette("Blues", n)
colors_hat = sns.color_palette("Reds", n)

# Time axis
x = np.linspace(0, 50, T)

# Plot each dimension in a separate subplot
for i in range(n):
    ax[i].plot(x, optimal_control_U_test[:, i], color=colors_true[i], linewidth=1, label=f'True Dim {i+1}')
    ax[i].plot(x, U_hat_LTI[:, i], color=colors_hat[i], linewidth=1, linestyle='--', label=f'Hat Dim {i+1}')
    ax[i].set_title(f'Dimension {i+1}')
    ax[i].spines['top'].set_visible(False)
    ax[i].spines['right'].set_visible(False)
    ax[i].legend(frameon=False, fontsize=8)

# Common labels and adjustments
fig.text(0.5, 0.04, 'Time', ha='center')
fig.text(0.04, 0.5, 'Value', va='center', rotation='vertical')
plt.tight_layout()
plt.show()

In [ ]:
trajectory_true = simulate_fractional_system_no_noise(A, B, alpha, trajectories_array_test[0], optimal_control_U_test, Time)

In [ ]:
T, n = trajectory_true.shape

# Define plot parameters
fig, ax = plt.subplots( 1, n, figsize=(15, 3), dpi=300, sharex=True, sharey=True)

# Define colors for trajectories
colors_true = sns.color_palette("Blues", n)
colors_hat = sns.color_palette("Reds", n)

# Time axis
x = np.linspace(0, 50, T)

# Plot each dimension in a separate subplot
for i in range(n):
    ax[i].plot(x, trajectory_true[:, i], color=colors_true[i], linewidth=1, label=f'True Dim {i+1}')
    ax[i].plot(x, trajectory_hat_LTI[:, i], color=colors_hat[i], linewidth=1, linestyle='--', label=f'Hat Dim {i+1}')
    ax[i].set_title(f'Dimension {i+1}')
    ax[i].spines['top'].set_visible(False)
    ax[i].spines['right'].set_visible(False)
    ax[i].legend(frameon=False, fontsize=8)

# Common labels and adjustments
fig.text(0.5, 0.04, 'Time', ha='center')
fig.text(0.04, 0.5, 'Value', va='center', rotation='vertical')
plt.tight_layout()
plt.show()

In [ ]:
mean_squared_error(optimal_control_U_test, U_hat_LTI)

In [ ]:
mean_squared_error(trajectory_true, trajectory_hat_LTI)